# Ice Cream Cone Demo

Interactive playground for VoxelCAD's unified geometry interface (Phase 10.5).

Covers: primitives, CSG booleans, affine transforms, PyVista rendering, profiling.

## Configuration

All tweakable settings in one place. Change these and re-run cells below.

In [ ]:
# === ENVIRONMENT SETTINGS ===
import voxelcad.environment as ENV

ENV.voxel_size = 0.15       # default voxel size (mm) - smaller = finer but slower
ENV.use_cython = True       # True: Cython+OpenMP kernels, False: NumPy fallback

# === PROFILING / LOGGING ===
import logging
logging.basicConfig(level=logging.WARNING)

# Per-module verbosity: set to logging.DEBUG to see render internals
logging.getLogger('voxelcad').setLevel(logging.WARNING)

# super_utils structured profiling (TIMING_START/TIMING_END labels in render paths)
from voxelcad.debug import HAS_SUPER_UTILS, TIMING_START, TIMING_END, TIMING_EXPORT_JSON
from voxelcad.debug import MEMORY_USAGE
PROFILING_ENABLED = True    # flip to False to skip timing cells

# === PYVISTA RENDERING ===
import os
os.environ['VTK_SILENCE_GET_VOID_POINTER_WARNINGS'] = '1'

import pyvista as pv
pv.global_theme.jupyter_backend = 'static'  # 'trame' for interactive rotation
OFFSCREEN = True            # True for inline images, False for interactive windows

# === DEMO PARAMETERS ===
VS = ENV.voxel_size
CONE_HEIGHT = 8.0
CONE_R_TOP = 3.0
SCOOP_RADIUS = 3.0

print(f"Cython: {ENV.use_cython} (super_utils: {HAS_SUPER_UTILS})")
print(f"Voxel size: {VS} mm")
print(f"PyVista backend: {pv.global_theme.jupyter_backend}")

In [ ]:
import time
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

import numpy as np
from voxelcad import Cube, Sphere, Cylinder, GyroidCube, union_all
from voxelcad.voxel_model import VoxelModel, CSGModel, TransformedModel
from voxelcad.voxel_grid import VoxelGrid

## 1. Primitives

Each primitive has `_render_cython` (fast) and `_render_numpy` (reference) implementations.
The unified interface `render_on_grid(grid, M4inv=None)` dispatches based on `ENV.use_cython`.

In [ ]:
# Cone: Cylinder with linear radius interpolation, point at bottom
cone = Cylinder(h=CONE_HEIGHT, r1=0, r2=CONE_R_TOP, center=True, voxel_size=VS)

# Scoop: Sphere intersected with GyroidCube for a textured surface
sphere = Sphere(r=SCOOP_RADIUS, voxel_size=VS)
gyroid = GyroidCube(size=SCOOP_RADIUS * 2, voxel_size=VS, center=True)
scoop = sphere & gyroid  # CSG intersection - gyroid-textured sphere
scoop_up = scoop.translate([0, 0, CONE_HEIGHT / 2])

print(f"Cone:     {type(cone).__name__:20s}  grid res = {cone.grid.res_vector.astype(int)}")
print(f"Sphere:   {type(sphere).__name__:20s}  grid res = {sphere.grid.res_vector.astype(int)}")
print(f"Gyroid:   {type(gyroid).__name__:20s}  grid res = {gyroid.grid.res_vector.astype(int)}")
print(f"Scoop:    {type(scoop).__name__:20s}  (sphere & gyroid)")
print(f"Scoop_up: {type(scoop_up).__name__:20s}  grid res = {scoop_up.grid.res_vector.astype(int)}")
print()
print(f"Cone-scoop grids same:       {cone.grid.same_grid(scoop_up.grid)}")
print(f"Cone-scoop grids compatible: {cone.grid.compatible_grid(scoop_up.grid)}")

## 2. CSG Booleans

Operators: `|` (union), `&` (intersection), `-` (difference), `^` (xor), `~` (invert).

When both operands share `voxel_size` but different grid bounds, the result is a lazy
`CSGModel`. Calling `render_volume()` triggers the query planner: all leaves render on
a common union grid, then byte-level postfix combination.

In [ ]:
# Union - lazy until render_volume()
ice_cream = cone | scoop_up
print(f"Union type: {type(ice_cream).__name__}")
print(f"Union grid: xlim={ice_cream.grid.xlim}, zlim={ice_cream.grid.zlim}")

# Materialize
t0 = time.perf_counter()
ice_cream.render_volume()
dt = time.perf_counter() - t0
print(f"\nrender_volume(): {dt*1000:.1f} ms")
print(f"Packed data: {ice_cream.voxel_data.shape[0]} bytes")

In [ ]:
# All four boolean operations
cube_a = Cube(size=6, voxel_size=VS)
sphere_b = Sphere(r=4, voxel_size=VS)

ops = [
    (cube_a | sphere_b, "A | B (union)"),
    (cube_a & sphere_b, "A & B (intersection)"),
    (cube_a - sphere_b, "A - B (difference)"),
    (cube_a ^ sphere_b, "A ^ B (xor)"),
]

for model, label in ops:
    t0 = time.perf_counter()
    model.render_volume()
    dt = time.perf_counter() - t0
    print(f"{label:30s}  type={type(model).__name__:12s}  {dt*1000:.1f} ms")

## 3. PyVista Rendering

In [ ]:
def make_mesh(model):
    """Render model and extract PyVista mesh."""
    if model.voxel_data is None:
        model.render_volume()
    return model.render_volume_mesh()

In [ ]:
pl = pv.Plotter(shape=(1, 3), off_screen=OFFSCREEN, window_size=(1800, 600))

parts = [
    (cone,      "Cone",                     'tan'),
    (scoop_up,  "Scoop (sphere & gyroid)",  'lightpink'),
    (ice_cream, "Ice Cream (union)",        'lightyellow'),
]

for i, (model, label, color) in enumerate(parts):
    pl.subplot(0, i)
    pl.add_text(label, font_size=14)
    pl.add_mesh(make_mesh(model), color=color, show_edges=False)
    pl.camera.azimuth = 30
    pl.camera.elevation = 15

pl.show()

## 4. Affine Transforms

Transforms are lazy `TransformedModel` wrappers. They store a 4x4 matrix pair (M4, M4inv)
and delegate to `source.render_on_grid(expanded_grid, M4inv)` at render time.

Chained transforms compose into a single matrix (no nesting).

In [ ]:
variants = [
    (ice_cream,                    "Original",         'lightyellow'),
    (ice_cream.rotate_x(25),       "Tilt X 25deg",     'lightblue'),
    (ice_cream.rotate_z(45),       "Rotate Z 45deg",   'lightgreen'),
    (ice_cream.scale([1.5, 0.5, 1.0]), "Scale [1.5,0.5,1]", 'lightsalmon'),
]

pl = pv.Plotter(shape=(1, 4), off_screen=OFFSCREEN, window_size=(2400, 600))

for i, (model, label, color) in enumerate(variants):
    t0 = time.perf_counter()
    mesh = make_mesh(model)
    dt = time.perf_counter() - t0
    pl.subplot(0, i)
    pl.add_text(f"{label}\n{dt*1000:.0f}ms", font_size=12)
    pl.add_mesh(mesh, color=color, show_edges=False)
    pl.camera.azimuth = 30
    pl.camera.elevation = 15

pl.show()

## 5. CSG Trees (Transforms + Booleans)

Combining transformed primitives in CSG. The query planner collects all leaves,
computes the union grid, and renders each leaf with its respective M4inv.

In [ ]:
# Rotated cube intersected with sphere - "diamond cut"
diamond_cut = Cube(size=8, voxel_size=VS).rotate_z(45) & Sphere(r=5, voxel_size=VS)

# Gyroid shell: gyroid - smaller gyroid
outer = GyroidCube(size=8, voxel_size=VS, center=True)
inner = GyroidCube(size=6, voxel_size=VS, center=True)
gyroid_shell = outer - inner

# Three-way union
combined = union_all([
    diamond_cut.translate([-10, 0, 0]),
    ice_cream,
    gyroid_shell.translate([10, 0, 0]),
])

t0 = time.perf_counter()
combined.render_volume()
dt = time.perf_counter() - t0
print(f"Three-model CSG tree: {dt*1000:.0f} ms")

pl = pv.Plotter(off_screen=OFFSCREEN, window_size=(1200, 600))
pl.add_mesh(make_mesh(combined), color='lightyellow', show_edges=False)
pl.camera.azimuth = 30
pl.camera.elevation = 20
pl.show()

## 6. Profiling

Compare Cython vs NumPy paths and measure render times across resolutions.

In [ ]:
if PROFILING_ENABLED:
    resolutions = [32, 64, 128, 256]
    
    print(f"{'res':>6s}  {'Cython (ms)':>12s}  {'NumPy (ms)':>12s}  {'Speedup':>8s}")
    print("-" * 44)
    
    for res in resolutions:
        vs_test = 10.0 / res
        
        # Cython path
        ENV.use_cython = True
        s = Sphere(r=4, voxel_size=vs_test)
        t0 = time.perf_counter()
        s.render_volume()
        dt_cy = time.perf_counter() - t0
        
        # NumPy path
        ENV.use_cython = False
        s2 = Sphere(r=4, voxel_size=vs_test)
        t0 = time.perf_counter()
        s2.render_volume()
        dt_np = time.perf_counter() - t0
        
        speedup = dt_np / dt_cy if dt_cy > 0 else float('inf')
        print(f"{res:>6d}  {dt_cy*1000:>12.1f}  {dt_np*1000:>12.1f}  {speedup:>7.1f}x")
    
    # Restore
    ENV.use_cython = True
    print(f"\nENV.use_cython restored to {ENV.use_cython}")
else:
    print("Profiling disabled (set PROFILING_ENABLED = True in config cell)")

In [ ]:
if PROFILING_ENABLED:
    mem_base = MEMORY_USAGE()
    
    # Render a moderately large model
    big = Sphere(r=5, voxel_size=0.05)
    big.render_volume()
    
    mem_after = MEMORY_USAGE()
    res_vec = big.grid.res_vector.astype(int)
    total_voxels = int(np.prod(res_vec))
    packed_bytes = big.voxel_data.shape[0]
    
    print(f"Grid: {res_vec[0]} x {res_vec[1]} x {res_vec[2]} = {total_voxels:,} voxels")
    print(f"Packed: {packed_bytes:,} bytes ({packed_bytes/1024:.1f} KB)")
    print(f"Bool equivalent: {total_voxels:,} bytes ({total_voxels/1024/1024:.1f} MB)")
    print(f"Compression: {total_voxels/packed_bytes:.0f}:1 (bit-packing)")
    print(f"RSS delta: {(mem_after - mem_base)/1024/1024:.1f} MB")

## 7. render_on_grid: The Unified Interface

Every VoxelModel can evaluate itself on any grid via `render_on_grid(grid, M4inv=None)`.
Primitives evaluate geometry; data-only models resample via nearest-neighbor.
This is what makes CSG trees work across heterogeneous leaf types.

In [ ]:
# A sphere rendered on its own grid vs a larger foreign grid
s = Sphere(r=3, voxel_size=VS)
s.render_volume()

foreign = VoxelGrid(xlim=(-5, 5), ylim=(-5, 5), zlim=(-5, 5), voxel_size=VS)

# Same grid shortcut - returns cached data
t0 = time.perf_counter()
packed_same = s.render_on_grid(s.grid)
dt_same = time.perf_counter() - t0

# Foreign grid - full re-evaluation (Cython or NumPy)
t0 = time.perf_counter()
packed_foreign = s.render_on_grid(foreign)
dt_foreign = time.perf_counter() - t0

print(f"Same grid (cached):   {dt_same*1000:.3f} ms  ({packed_same.shape[0]} bytes)")
print(f"Foreign grid (eval):  {dt_foreign*1000:.1f} ms  ({packed_foreign.shape[0]} bytes)")
print(f"\nSame-grid shortcut is {dt_foreign/max(dt_same, 1e-9):.0f}x faster (returns cached data)")

In [ ]:
# Data-only model (base VoxelModel) uses resampling as its "geometry evaluation".
# The ice_cream CSGModel, once rendered, becomes a data-only model.
# Resampling it onto a different grid uses the resample_and_pack Cython kernel.

target_grid = VoxelGrid(
    xlim=(-5, 5), ylim=(-5, 5), zlim=(-6, 10), voxel_size=VS
)

t0 = time.perf_counter()
resampled = ice_cream.render_on_grid(target_grid)
dt_resample = time.perf_counter() - t0

print(f"Resampled ice_cream onto foreign grid: {dt_resample*1000:.1f} ms")
print(f"Source packed: {ice_cream.voxel_data.shape[0]} bytes")
print(f"Target packed: {resampled.shape[0]} bytes")

## 8. Sandbox

Try your own geometry below.

In [ ]:
# Build something here
# my_model = Cube(10, voxel_size=0.2) & Sphere(r=6, voxel_size=0.2).rotate_z(45)
# my_model.render_volume()
# pl = pv.Plotter(off_screen=OFFSCREEN)
# pl.add_mesh(make_mesh(my_model), color='lightblue')
# pl.show()